# Brave Search API — value comparison demo

A live walkthrough comparing three ways to answer questions about recent events:

| Pipeline | What it does |
| --- | --- |
| **baseline** | LLM only, no retrieval — the hallucination baseline |
| **diy_rag** | Manual RAG (search + fetch + extract + chunk + embed + index + retrieve) |
| **brave-search-api** | Brave LLM Context endpoint → LLM with citation enforcement |

This notebook tells two stories at once:

1. **Quality** — `baseline` vs `brave-search-api`. Does retrieval reduce hallucinations on questions about recent events?
2. **Infrastructure** — `diy_rag` vs `brave-search-api`. If you build RAG from scratch instead of using Brave's LLM Context endpoint, how much extra code, latency, and dependency surface do you take on — and do you get better answers for the trouble?

LangFuse traces every call, scores every output, and produces the comparison view at the end.


## 1. Setup

Verify env vars and initialize clients. Required vars: `BRAVE_API_KEY`, `ANTHROPIC_API_KEY`, `LANGFUSE_HOST`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`. Set them in your environment however you manage secrets, *before* starting Jupyter.

LangFuse runs **locally** via the repo's `docker-compose.yml` (`docker compose up -d`). It seeds a project and a fixed key pair on first boot — no UI setup — so just export the seeded values:

```
LANGFUSE_HOST=http://localhost:3005
LANGFUSE_PUBLIC_KEY=pk-lf-local-brave-demo
LANGFUSE_SECRET_KEY=sk-lf-local-brave-demo
```


In [ ]:
import os

required = ["BRAVE_API_KEY", "ANTHROPIC_API_KEY", "LANGFUSE_HOST", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise EnvironmentError(
        f"Missing env vars: {missing}. "
        "Set them in your environment before launching Jupyter."
    )
print("✓ All required env vars present")
print(f"  LANGFUSE_HOST: {os.environ['LANGFUSE_HOST']}")
print(f"  ANSWER_MODEL:  {os.environ.get('ANSWER_MODEL', 'claude-opus-4-7 (default)')}")
print(f"  JUDGE_MODEL:   {os.environ.get('JUDGE_MODEL',  'claude-opus-4-7 (default)')}")


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

from langfuse import get_client

langfuse = get_client()
print(f"✓ LangFuse client ready  (host: {os.environ.get('LANGFUSE_HOST', 'http://localhost:3005')})")


## 2. The three pipelines

Each is defined in its own file under `pipelines/`. The size difference is part of the story — same input/output contract, dramatically different implementation cost.


In [ ]:
# Three pipelines, same interface: takes a question string, returns
# {"answer": str, "sources": list[dict], ...}
from pipelines import baseline_pipeline, diy_rag_pipeline, brave_search_api_pipeline
from pipelines import diy_rag  # for pre-warming the embedder later

print("✓ Pipelines imported")


In [ ]:
# Line counts (the value prop in a directory listing)
import pathlib

for name in ("baseline.py", "diy_rag.py", "brave_search_api.py"):
    lines = len(pathlib.Path("pipelines", name).read_text().splitlines())
    print(f"  {name:<20} {lines:>4} lines")


## 3. Live demo — one question, three pipelines in parallel

Before running the full LangFuse eval, let's see all three pipelines answer a single question side-by-side. This is the visceral moment: real answers, real latency, all in parallel.

We pre-warm the local embedder *off the clock* so DIY's reported time reflects steady-state, not first-load.


In [ ]:
DEMO_QUESTION = (
    "What was the Federal Reserve's most recent interest rate decision in 2026, "
    "and what reasoning did Powell give?"
)
print(f"Question: {DEMO_QUESTION}")


In [ ]:
# Pre-warm — this is setup, not the timed run. First call downloads ~80MB,
# subsequent calls are cached. Print "done" when complete.
print("Pre-warming local embedder...", end=" ", flush=True)
t0 = time.perf_counter()
diy_rag._get_embedder()
print(f"done in {time.perf_counter() - t0:.1f}s")


In [ ]:
# Run all three pipelines IN PARALLEL on the same question.
# Wall-clock time = the slowest pipeline (DIY), not the sum.

def run_one(name, fn, question):
    start = time.perf_counter()
    try:
        result = fn(question)
        result["_ok"] = True
    except Exception as e:
        result = {"answer": f"ERROR: {e}", "sources": [], "_ok": False}
    result["_latency_s"] = time.perf_counter() - start
    result["_name"] = name
    return result

pipelines_list = [
    ("baseline",         baseline_pipeline),
    ("diy_rag",          diy_rag_pipeline),
    ("brave-search-api", brave_search_api_pipeline),
]

wall_start = time.perf_counter()
results = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(run_one, n, fn, DEMO_QUESTION): n for n, fn in pipelines_list}
    for fut in as_completed(futures):
        r = fut.result()
        results[r["_name"]] = r
        print(f"  ✓ {r['_name']:<16} returned at +{r['_latency_s']:.1f}s")

print(f"\nWall-clock (parallel): {time.perf_counter() - wall_start:.1f}s")


In [ ]:
# Display each answer
for name in ("baseline", "diy_rag", "brave-search-api"):
    r = results[name]
    print("=" * 70)
    print(f"{name.upper()}  ({r['_latency_s']:.1f}s)")
    print("=" * 70)
    print(r["answer"])
    if r.get("sources"):
        print(f"\n  Sources cited: {len(r['sources'])}")
    print()


Three things to look for in the output above:

- **baseline** likely hedges, vagues, or invents specifics — no sources to ground in.
- **diy_rag** should have inline `[1]`, `[2]` citations, but may be visibly slower and sometimes worse (chunking artifacts, extraction failures).
- **brave-search-api** should also have inline `[1]`, `[2]` citations with specific facts — with far less machinery.

That's the qualitative reveal. Now let's quantify it across a small dataset and let LangFuse score it.


## 4. Push the eval dataset to LangFuse

Five hand-picked finance questions across categories (central bank, earnings, macro data, corporate, refusal test). Small enough to live-run, varied enough to be meaningful.


In [ ]:
DATASET_NAME = "brave-finance-eval"

QUESTIONS = [
    {"question": "What was the Federal Reserve's most recent interest rate decision in 2026, and what reasoning did Powell give?", "category": "central_bank"},
    {"question": "What were Nvidia's most recent quarterly earnings results — revenue, EPS, and guidance?", "category": "earnings"},
    {"question": "What was the most recent US CPI inflation reading released in 2026?", "category": "macro_data"},
    {"question": "What major tech-sector M&A deals were announced in early 2026?", "category": "corporate"},
    {"question": "What will the Federal Reserve decide at its next meeting after May 2026?", "category": "refusal_test"},
]

try:
    langfuse.create_dataset(
        name=DATASET_NAME,
        description="Finance Q&A on recent events. baseline vs diy_rag vs brave-search-api.",
    )
    print(f"Created dataset '{DATASET_NAME}'.")
except Exception as e:
    print(f"Dataset already exists (or: {e}). Continuing.")

for q in QUESTIONS:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input={"question": q["question"]},
        metadata={"category": q["category"]},
    )

langfuse.flush()
print(f"Uploaded {len(QUESTIONS)} items.")


## 5. The eval policy — what we're measuring

Three scores attached to every dataset item run:

- **`citation_rate`** (0–1) — LLM-as-judge: what fraction of factual claims have inline citations?
- **`factuality`** (1–5) — LLM-as-judge: how substantive and grounded is the answer?
- **`latency_ms`** — measured during execution

The judges and the task wrappers live in `evaluation.py` as library code. Keeping them there lets the notebook stay focused on the demo narrative — importing 70 lines of judge prompts is less interesting than importing them and inspecting one. Let's do exactly that.


In [ ]:
from evaluation import (
    EVALUATORS,                        # the three evaluators packaged up
    baseline_task, diy_task, brave_search_api_task,  # task wrappers (handle latency tracking)
    citation_rate_evaluator,           # individual evaluators for inspection
    factuality_evaluator,
    latency_evaluator,
)

print(f"✓ Loaded {len(EVALUATORS)} evaluators: " + ", ".join(e.__name__ for e in EVALUATORS))


In [ ]:
# Show what the citation judge actually does. This is the policy
# that determines our headline numbers — worth showing the audience.
import inspect
from evaluation import JUDGE_CITATION

print("Citation rate judge prompt:")
print("-" * 70)
print(JUDGE_CITATION)


## 6. Run the three experiments

Each experiment is one pipeline run against the dataset, with all evaluators applied. `max_concurrency` lets items run in parallel within an experiment — the wall-clock for each is roughly the slowest single item's runtime.


In [ ]:
dataset = langfuse.get_dataset(DATASET_NAME)
n_items = len(list(dataset.items))
RUN_TAG = "v1"  # bump this between rerun attempts
ANSWER_MODEL = os.environ.get("ANSWER_MODEL", "claude-opus-4-7")
print(f"Dataset '{DATASET_NAME}': {n_items} items.  RUN_TAG = '{RUN_TAG}'")


In [ ]:
# Experiment 1: baseline — LLM only, no retrieval
baseline_result = dataset.run_experiment(
    name=f"baseline-{RUN_TAG}",
    description="LLM only, no retrieval. Hallucination baseline.",
    task=baseline_task,
    evaluators=EVALUATORS,
    max_concurrency=5,
    metadata={"model": ANSWER_MODEL, "pipeline": "baseline"},
)
print(baseline_result.format())


In [ ]:
# Experiment 2: diy_rag — manual RAG pipeline
diy_result = dataset.run_experiment(
    name=f"diy-rag-{RUN_TAG}",
    description=(
        "Manual RAG: Brave Web Search + trafilatura extraction + paragraph chunking + "
        "sentence-transformers embeddings + FAISS retrieval. Same citation-forcing prompt."
    ),
    task=diy_task,
    evaluators=EVALUATORS,
    max_concurrency=3,  # local embedder + concurrent fetches: less is more
    metadata={
        "model": ANSWER_MODEL,
        "pipeline": "diy_rag",
        "embedding_model": "all-MiniLM-L6-v2",
        "vector_store": "faiss-IndexFlatIP",
    },
)
print(diy_result.format())


In [ ]:
# Experiment 3: brave-search-api — Brave LLM Context
brave_search_api_result = dataset.run_experiment(
    name=f"brave-search-api-{RUN_TAG}",
    description="Brave LLM Context endpoint + citation-forcing prompt.",
    task=brave_search_api_task,
    evaluators=EVALUATORS,
    max_concurrency=5,
    metadata={"model": ANSWER_MODEL, "pipeline": "brave-search-api", "freshness": "pm"},
)
langfuse.flush()
print(brave_search_api_result.format())


## 7. The comparison view — this is the money slide

Open LangFuse → Datasets → `brave-finance-eval` → Runs → select all three → **Compare**.

Look at:

- **Aggregate `factuality`** — baseline near 2/5, diy_rag in between, brave-search-api near 4+
- **Aggregate `citation_rate`** — baseline near 0, diy_rag in between, brave-search-api near 90%
- **Aggregate `latency_ms`** — diy_rag ~15s+, brave-search-api ~3s
- **One trace** — click any item to see the full chain (Brave call → LLM call → output with citations)


In [ ]:
host = os.environ.get("LANGFUSE_HOST", "http://localhost:3005")
print(f"Open this URL in your browser:\n  {host}\n")
print("Then navigate: Datasets → brave-finance-eval → Runs → select all three → Compare")


## 8. The code contrast — the closer

LangFuse showed you that brave-search-api beats baseline on quality, and beats diy_rag on latency. The code contrast shows you what those wins cost in engineering effort.


In [ ]:
# Show the dependency declarations from pyproject.toml.
# TOML preserves the inline comments grouping shared vs DIY-only deps.
import pathlib, re
text = pathlib.Path("pyproject.toml").read_text()
match = re.search(r"(dependencies = \[.*?\])", text, re.DOTALL)
print(match.group(1) if match else text)


In [ ]:
# The DIY pipeline — eight numbered steps, three additional heavy dependencies.
# This is what Brave's LLM Context endpoint collapses into a single API call.
print(inspect.getsource(diy_rag_pipeline))


In [ ]:
# The brave-search-api pipeline — ~30 lines of orchestration, one external API call to Brave.
# This is what the entire DIY stack above collapses into.
print(inspect.getsource(brave_search_api_pipeline))


## 9. Takeaways

**Quality:** Brave Search API answers had inline citations, specific facts, and appropriate refusal on unanswerable questions. The baseline had none of those.

**Infrastructure:** The brave-search-api pipeline is ~30 lines, one API call, two dependencies. The diy_rag pipeline is ~280 lines, eight steps, six dependencies, ~5× slower. The Brave LLM Context endpoint replaces an entire retrieval stack with one HTTP call.

**What this notebook is not:** a production benchmark. The dataset is 5 questions. Judge scoring is non-deterministic. For rigorous evaluation, run multiple variants with different `RUN_TAG` values and average across runs.

**Iteration paths from here:**

- Bump `RUN_TAG = "v2"` and rerun any experiment with different parameters (`count`, `freshness`, `context_threshold_mode` on the brave-search-api side; chunk size, embedder, retrieval `k` on the DIY side). LangFuse keeps each as a separate column in the comparison view.
- Add more dataset items by re-running the dataset cell with additional `QUESTIONS`.
- Swap the answer model (`ANSWER_MODEL` env var) to test Claude vs GPT vs an open model holding the retrieval pipeline constant.

**Repo layout:**

```
walkthrough.ipynb     ← you are here
evaluation.py         ← judges + task wrappers (library, imported above)

pipelines/
    baseline.py          ← 32 lines  — LLM only
    diy_rag.py           ← 280 lines — manual RAG
    brave_search_api.py  ← 110 lines — Brave LLM Context
```
